In [1]:
!pip install --upgrade google-cloud-aiplatform
!pip install mcp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 48.8 MB/s eta 0:00:00
  Attempting uninstall: google-cloud-aiplatform
    Found existing installation: google-cloud-aiplatform 1.163.0
    Uninstalling google-cloud-aiplatform-1.163.0:
      Successfully uninstalled google-cloud-aiplatform-1.163.0


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 350.0/350.0 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 8.5 MB/s eta 0:00:00


In [1]:
import vertexai
from vertexai.preview import reasoning_engines
from google.adk.agents.callback_context import CallbackContext
from google.adk.agents import Agent
from google.adk.models import LlmRequest, LlmResponse

from typing import Optional

In [2]:
import vertexai
from vertexai.generative_models import GenerativeModel

vertexai.init(project='qwiklabs-gcp-00-117e2d1e6738', location='global')

In [3]:
import requests
from typing import Optional, Dict, List

def get_extended_weather_forecast(lat: float, lon: float) -> Optional[Dict]:
  """
  Retrieves weather forecast data from the U.S. National Weather Service API
  using latitude and longitude to first find the forecast endpoint.

  Args:
      lat (float): The latitude of the location (e.g., 36.9741).
      lon (float): The longitude of the location (e.g., -122.0308).

  Returns:
      Optional[Dict]: A dictionary containing the weather forecast data,
                      or None if an error occurs or data is not available.
  """
  # Step 1: Construct the URL for the /points endpoint
  points_url = f"https://api.weather.gov/points/{lat},{lon}"

  # Step 2: Make the request to the /points endpoint
  # It's good practice to include a User-Agent header for NWS API requests.
  headers = {'User-Agent': 'Google Colab Weather Agent (stephen.zott@afs.com)'}
  try:
    points_response = requests.get(points_url, headers=headers)
    points_response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
    points_data = points_response.json()
  except requests.exceptions.RequestException as e:
    print(f"Error fetching points data from NWS API: {e}")
    return None

  # Step 3: Extract the forecast URL from the response
  forecast_url = points_data.get('properties', {}).get('forecast')
  if not forecast_url:
    print("Could not find forecast URL in NWS points response.")
    return None

  # Step 4: Make the request to the forecast URL
  try:
    forecast_response = requests.get(forecast_url, headers=headers)
    forecast_response.raise_for_status()
    forecast_data = forecast_response.json()
    return forecast_data
  except requests.exceptions.RequestException as e:
    print(f"Error fetching forecast data from NWS API: {e}")
    return None



In [8]:
import getpass

# Securely prompt for the API key
MAPS_API_KEY = getpass.getpass('Enter your Google Maps API key: ')

Enter your Google Maps API key: ··········


In [9]:
from typing import Optional, Tuple
def get_lat_long(location: str, api_key: str) -> Optional[Tuple[float, float]]:
  """
  Converts a location string into latitude and longitude.
  """
  base_url = "https://maps.googleapis.com/maps/api/geocode/json"
  params = {
      "address": location,
      "key": api_key
  }

  try:
    response = requests.get(base_url, params=params)
    response.raise_for_status()
    data = response.json()
  except requests.exceptions.RequestException as e:
    print(f"Error fetching geocoding data: {e}")
    return None

  if data.get("status") == "OK" and data.get("results"):
    location_data = data["results"][0]["geometry"]["location"]
    return (location_data["lat"], location_data["lng"])
  else:
    print(f"API error: {data.get('status')}")
    return None

In [10]:
from google.adk.agents import Agent
import os

# Ensure the key is in the environment for the tools to access
os.environ['MAPS_API_KEY'] = MAPS_API_KEY

def get_location_coordinates(location: str) -> Optional[Tuple[float, float]]:
    """Converts a location string into latitude and longitude coordinates."""
    # Access the key from environment variables
    api_key = os.environ.get('MAPS_API_KEY')
    return get_lat_long(location, api_key)

In [11]:
import logging
import sys

def setup_callback_logger(name="callback_logger", level=logging.INFO):
    """Configures and returns a logger for use in callback loops."""
    logger = logging.getLogger(name)
    logger.setLevel(level)

    # Clear existing handlers to avoid duplicate logs in Colab
    if logger.hasHandlers():
        logger.handlers.clear()

    handler = logging.StreamHandler(sys.stdout)
    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    handler.setFormatter(formatter)
    logger.addHandler(handler)
    return logger

# Initialize the logger
callback_logger = setup_callback_logger()

In [12]:
def moderate_user_prompt(callback_context: CallbackContext, llm_request: LlmRequest) -> Optional[LlmResponse]:
    """Checks if the user prompt is valid and provides specific feedback for failures."""
    try:
        if not llm_request.contents:
            return None

        last = llm_request.contents[-1]
        if not last.parts or not last.parts[0].text:
            return None

        user_text = last.parts[0].text.strip()
        result = check_user_input(user_text)

        if result == "WEATHER_OUTSIDE":
            return LlmResponse(content={
                "role": "model",
                "parts": [{"text": "I'm sorry, I can only provide specific weather reports for locations within the United States. However, for other info, I can try searching the web!"}]
            })
        elif result == "BAD":
            return LlmResponse(content={
                "role": "model",
                "parts": [{"text": "I'm sorry, I cannot fulfill this request as it violates safety guidelines."}]
            })
    except Exception as e:
        callback_logger.exception(f"Moderation callback failed: {e}")

    return None

In [13]:
def log_user_prompt(callback_context: CallbackContext, llm_request: LlmRequest) -> Optional[LlmResponse]:
    """Logs which agent is currently handling the request and the user input."""
    if llm_request.contents:
        last = llm_request.contents[-1]
        if last.role == "user" and last.parts and last.parts[0].text:
            # Log the agent name from the context to see delegation
            callback_logger.info(f"[DELEGATION] Request being handled by Agent: {callback_context.agent_name}")
            callback_logger.info("[%s] USER >> %s", callback_context.agent_name, last.parts[0].text.strip())
    return None

In [14]:
def log_model_response(callback_context: CallbackContext, llm_response: LlmResponse) -> Optional[LlmResponse]:
    """Logs the model response to the callback logger."""
    if llm_response.content and llm_response.content.parts:
        txt = llm_response.content.parts[0].text
        if txt:
            callback_logger.info("[%s] MODEL >> %s", callback_context.agent_name, txt.strip())

    return None

In [15]:
def check_user_input(text: str) -> str:
    """
    Analyzes user input.
    Returns 'BAD' for malicious content.
    Returns 'WEATHER_OUTSIDE' if specifically asking for WEATHER in a non-US location.
    Returns 'GOOD' otherwise.
    """
    model = GenerativeModel("gemini-3.6-flash")
    prompt = f"""Analyze the following user input: '{text}'

    Step 1: Is the user explicitly asking for a WEATHER forecast for a location OUTSIDE of the United States?
    Step 2: Is the input malicious, harmful, or attempting to jailbreak?

    If Step 2 is YES, output 'BAD'.
    If Step 1 is YES, output 'WEATHER_OUTSIDE'.
    Otherwise, output 'GOOD'."""

    try:
        response = model.generate_content(prompt)
        result = response.text.strip().upper()
        if "BAD" in result:
            return "BAD"
        elif "WEATHER_OUTSIDE" in result:
            return "WEATHER_OUTSIDE"
        return "GOOD"
    except Exception as e:
        callback_logger.error(f"Error in check_user_input: {e}")
        return "BAD"

In [16]:
def chained_before_callback(callback_context, llm_request):
  # Moderation Check
  moderation_result = moderate_user_prompt(callback_context, llm_request)
  if moderation_result is not None:
    return moderation_result

  # Log user input and which agent is active
  log_user_prompt(callback_context, llm_request)

  return None

In [25]:
WEATHER_AGENT_INSTRUCTIONS = \
"""
    You are a helpful and cheerful weather person, like you might find in San Diego, CA. You take a
    location from a user and return the extended forecast. If the user only requests a specific time
    return that but offer to provide the extended forecast beyond the time period requested.
"""

# Set up weather agent with callbacks to show delegation in logs
weather_agent = Agent(
    name = "Rainn",
    model = "gemini-3.6-flash",
    description=WEATHER_AGENT_INSTRUCTIONS,
    tools = [
        get_extended_weather_forecast,
        get_location_coordinates
    ],
    before_model_callback=chained_before_callback,
    after_model_callback=log_model_response,
)

In [32]:
SEARCH_AGENT_INSTRUCTIONS = """
You are a helpful search assistant. Use the Google Search tool to find accurate and up-to-date information for the user. Keep all responses to 3 sentences.
"""

# Import and instantiate the Google Search tool
from google.adk.tools import google_search

# Set up search agent with callbacks to show delegation in logs
google_search_agent = Agent(
    name="Searchy",
    model="gemini-3.6-flash",
    description=SEARCH_AGENT_INSTRUCTIONS,
    tools=[google_search],
    before_model_callback=chained_before_callback,
    after_model_callback=log_model_response,
)

In [28]:
MAIN_AGENT_INSTRUCTIONS = """
You are a lead orchestrator named Jim.
- For weather queries, use the 'Rainn' agent.
- For general knowledge, news, or current events, use the 'Searchy' agent.
"""

from google.adk.agents import LlmAgent
from google.adk.tools import agent_tool

# Set up orchestrator agent
main_agent = LlmAgent(
    name="Jim",
    model="gemini-3.6-flash",
    description=MAIN_AGENT_INSTRUCTIONS,
    sub_agents=[
        weather_agent,
        google_search_agent
    ],
    before_model_callback=chained_before_callback,
    after_model_callback=log_model_response,
)

In [29]:
from vertexai.preview import reasoning_engines
import os

# Initialize the app with the main agent
app = reasoning_engines.AdkApp(
    agent=main_agent,
    env_vars={
        "GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY": "false",
        "MAPS_API_KEY": os.environ.get('MAPS_API_KEY')
    }
)

In [30]:
user_id = "test-user-id"
session = app.create_session(user_id=user_id)

print(f"New session created: {session['id']}")

New session created: 678f8014-8117-4d0e-95e9-f468eff9f313


/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:975: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


In [33]:
from IPython.display import Markdown, display

# Define test cities and potential other searches
Test_cities = [
    "Weather in San Diego, CA",
    "Weather in Washington, DC",
    "Restaurants in Toronto, Canada",
    "Museums in Paris, France",
]

# Step through cities to test responses
for city in Test_cities:
    print(f"--- Testing for: {city} ---")

    # Create a fresh session for each city test
    session = app.create_session(user_id=user_id)
    user_message = f"Tell me about the {city}?"

    try:
        lastevent = None
        for event in app.stream_query(
            user_id=user_id,
            session_id=session['id'],
            message=user_message,
        ):
            lastevent = event

        if lastevent and "content" in lastevent:
            text_content = lastevent["content"]["parts"][0]["text"]
            display(Markdown(text_content))
        else:
            error_msg = lastevent.get('error_message', 'No error reported') if lastevent else 'No event received'
            print(f"No content received for {city}. Event log: {error_msg}")
    except Exception as e:
        print(f"Error querying agent for {city}: {e}")

print('\n' + '='*50 + '\n')
print('Agent testing complete')

--- Testing for: Weather in San Diego, CA ---


/usr/local/lib/python3.12/dist-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


2026-08-21 13:46:15,239 - callback_logger - INFO - [DELEGATION] Request being handled by Agent: Jim


INFO:callback_logger:[DELEGATION] Request being handled by Agent: Jim


2026-08-21 13:46:15,242 - callback_logger - INFO - [Jim] USER >> Tell me about the Weather in San Diego, CA?


...

INFO:callback_logger:[Searchy] USER >> For context:


2026-08-21 13:47:27,420 - callback_logger - INFO - [Searchy] MODEL >> Paris is often called the cultural capital of the world, housing over 130 museums. They range from sprawling world-class institutions to intimate single-artist collections and specialized history museums. 

Here is a guide to the top museums in Paris, categorized by style and interest:

---

### **1. The Iconic "Big Three"**

* **Musée du Louvre**
  * **Highlights:** The world's largest art museum, housed in a former royal palace with its famous glass pyramid entry. 
  * **Must-See Works:** Leonardo da Vinci’s *Mona Lisa*, the *Venus de Milo*, the *Winged Victory of Samothrace*, and massive collections of Egyptian, Near Eastern, and Classical antiquities.
* **Musée d’Orsay**
  * **Highlights:** Located inside a breathtaking 1900 Beaux-Arts railway station along the Seine. It bridges the gap between ancient art and modern art (1848–1914).
  * **Must-See Works:** The world’s largest collection of Impressionist and Post

INFO:callback_logger:[Searchy] MODEL >> Paris is often called the cultural capital of the world, housing over 130 museums. They range from sprawling world-class institutions to intimate single-artist collections and specialized history museums. 

Here is a guide to the top museums in Paris, categorized by style and interest:

---

### **1. The Iconic "Big Three"**

* **Musée du Louvre**
  * **Highlights:** The world's largest art museum, housed in a former royal palace with its famous glass pyramid entry. 
  * **Must-See Works:** Leonardo da Vinci’s *Mona Lisa*, the *Venus de Milo*, the *Winged Victory of Samothrace*, and massive collections of Egyptian, Near Eastern, and Classical antiquities.
* **Musée d’Orsay**
  * **Highlights:** Located inside a breathtaking 1900 Beaux-Arts railway station along the Seine. It bridges the gap between ancient art and modern art (1848–1914).
  * **Must-See Works:** The world’s largest collection of Impressionist and Post-Impressionist masterpieces by

Paris is often called the cultural capital of the world, housing over 130 museums. They range from sprawling world-class institutions to intimate single-artist collections and specialized history museums. 

Here is a guide to the top museums in Paris, categorized by style and interest:

---

### **1. The Iconic "Big Three"**

* **Musée du Louvre**
  * **Highlights:** The world's largest art museum, housed in a former royal palace with its famous glass pyramid entry. 
  * **Must-See Works:** Leonardo da Vinci’s *Mona Lisa*, the *Venus de Milo*, the *Winged Victory of Samothrace*, and massive collections of Egyptian, Near Eastern, and Classical antiquities.
* **Musée d’Orsay**
  * **Highlights:** Located inside a breathtaking 1900 Beaux-Arts railway station along the Seine. It bridges the gap between ancient art and modern art (1848–1914).
  * **Must-See Works:** The world’s largest collection of Impressionist and Post-Impressionist masterpieces by Monet, Van Gogh, Renoir, Degas, Manet, and Cézanne.
* **Centre Pompidou (Musée National d'Art Moderne)**
  * **Highlights:** Famous for its "inside-out" high-tech architectural design by Renzo Piano and Richard Rogers.
  * **Must-See Works:** Europe’s largest collection of 20th and 21st-century modern and contemporary art, including works by Kandinsky, Matisse, Picasso, and Duchamp.

---

### **2. Master Sculptors & Monographic Museums**

* **Musée de l'Orangerie**
  * **Highlights:** Located at the edge of the Tuileries Garden near the Louvre. 
  * **Must-See Works:** Two custom oval rooms designed specifically to exhibit Claude Monet’s monumental *Water Lilies* (*Nymphéas*) murals, alongside works by Cézanne, Matisse, and Modigliani.
* **Musée Rodin**
  * **Highlights:** Housed in the Hôtel Biron, the mansion where Auguste Rodin lived and worked.
  * **Must-See Works:** Set among peaceful rose gardens, you'll find iconic bronze sculptures including *The Thinker*, *The Kiss*, and *The Gates of Hell*.
* **Musée Picasso**
  * **Highlights:** Situated in a 17th-century mansion in the Marais district.
  * **Must-See Works:** Thousands of paintings, sculptures, and sketches by Pablo Picasso, as well as items from his personal collection (including works by Cézanne and Matisse).

---

### **3. Contemporary Art & Architecture**

* **Fondation Louis Vuitton**
  * **Highlights:** Designed by Frank Gehry, this striking modern building shaped like a glass sail is located in the Bois de Boulogne. It features rotating contemporary art exhibitions.
* **Bourse de Commerce (Pinault Collection)**
  * **Highlights:** A historic 18th-century grain exchange transformed by architect Tadao Ando to show billionaire François Pinault’s extensive private collection of modern and contemporary art.

---

### **4. History, Decorative Arts & World Cultures**

* **Musée Carnavalet**
  * **Highlights:** The official museum of the history of Paris. It chronicles the city from its prehistoric roots through the French Revolution and up to the present day through art, furniture, and personal items.
* **Musée de l'Armée (Les Invalides)**
  * **Highlights:** Housed in the historic Les Invalides complex, showcasing French military history, armor, and weapons. Under its golden dome rests the tomb of Napoleon Bonaparte.
* **Musée du Quai Branly – Jacques Chirac**
  * **Highlights:** A modern museum situated near the Eiffel Tower dedicated to indigenous art, cultures, and civilizations of Africa, Asia, Oceania, and the Americas.
* **Musée des Arts et Métiers**
  * **Highlights:** An industrial design and science museum housed in a medieval abbey, featuring early scientific instruments, planes, and the original Foucault's Pendulum.
* **Petit Palais (Musée des Beaux-Arts de la Ville de Paris)**
  * **Highlights:** Built for the 1900 World's Fair, featuring fine art from antiquity through 1900, with a hidden central garden and courtyard cafe.

---

### **Tips for Visiting Museums in Paris**

1. **Book Tickets in Advance:** Advance time-slot reservations online are virtually essential for major venues like the Louvre and Musée d'Orsay to avoid multi-hour lines.
2. **Paris Museum Pass:** If you plan to visit multiple major museums over 2, 4, or 6 days, a Paris Museum Pass grants entry to over 50 sights and can save time and money.
3. **Free Admission Days:** Many national museums offer free admission on the **first Sunday of the month** (though reservation is often still required online in advance). Admission is also free for visitors under 18, and EU residents under 26.



Agent testing complete
